In [3]:
### Bibliotecas
import pandas as pd
import numpy as np

In [4]:
#criar variavrel para o caminho do arquivo
caminho_arquivo="Base_Clinica_Odontologica_Felipe.xlsx"

In [6]:
# Ler o arquivo Excel
Excel_file = pd.ExcelFile(caminho_arquivo)  

In [14]:
Aba_Excel=Excel_file.sheet_names
print(f"Abas encontradas na planilha: {Aba_Excel}")

Abas encontradas na planilha: ['Pacientes', 'Dentistas', 'Atendimentos']


In [15]:
#Dicionário para armazenar os DataFrames tratados de cada aba
dados_tratados = {}

In [17]:
# FUNÇÃO GENÉRICA DE LIMPEZA E PADRONIZAÇÃO DE COLUNAS
# Finalidade: Remover espaços extras, padronizar nomes de colunas em minúsculo 
# e remover caracteres especiais/acentos para facilitar consultas.
def padronizar_colunas(df):
    """
    Remove espaços das extremidades, converte para minúsculo e
    substitui espaços internos por underline (_).
    """
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
        .str.replace('ã', 'a')
        .str.replace('á', 'a')
        .str.replace('é', 'e')
        .str.replace('í', 'i')
        .str.replace('ó', 'o')
        .str.replace('ú', 'u')
        .str.replace('ç', 'c')
    )
    return df

In [ ]:
#  TRATAMENTO ABA POR ABA
# Finalidade: Iterar por todas as abas, tratando tipos de dados, duplicadas,
# valores nulos e formatações específicas de texto/data/moeda.

for nome_aba in Aba_Excel:
    print(f"\n---> Iniciando tratamento da aba: '{nome_aba}'")
    
    # Leitura da aba atual
    df = pd.read_excel(Excel_file, sheet_name=nome_aba)
    
    #  Remoção de linhas e colunas completamente vazias
    # Finalidade: Eliminar sujeiras comuns de planilhas Excel.
    df = df.dropna(how='all').dropna(how='all', axis=1)
    
    #  Padronização do nome das colunas
    # Finalidade: Manter consistência técnica em todo o script.
    df = padronizar_colunas(df)
    
    # 3 Remoção de registros duplicados
    # Finalidade: Garantir a integridade dos dados e evitar contagem dupla.
    linhas_antes = len(df)
    df = df.drop_duplicates()
    linhas_depois = len(df)
    if linhas_antes - linhas_depois > 0:
        print(f"   [!] Foram removidas {linhas_antes - linhas_depois} linhas duplicadas.")
    
    #  Tratamento de Tipos de Dados e Textos
    # Finalidade: Garantir que datas, valores numéricos e textos estejam nos tipos corretos.
    for col in df.columns:
        # Se for texto/objeto, remove espaços em branco extras no início e fim
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.strip()
            
            # Substitui strings vazias/'nan' resultantes por NaN do NumPy
            df[col] = df[col].replace(['nan', 'NaN', 'None', ''], np.nan)
        
        # Conversão automática de colunas com nome contendo 'data' ou 'date' para datetime
        if 'data' in col or 'date' in col:
            df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
            
        # Conversão automática de valores financeiros ou numéricos com vírgula para float
        if 'valor' in col or 'preco' in col or 'custo' in col or 'total' in col:
            if df[col].dtype == 'object':
                df[col] = (
                    df[col]
                    .str.replace('R$', '', regex=False)
                    .str.replace('.', '', regex=False)
                    .str.replace(',', '.', regex=False)
                    .str.strip()
                )
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Armazena a aba tratada no dicionário
    dados_tratados[nome_aba] = df
    print(f"   [✓] Aba '{nome_aba}' tratada com sucesso! Formato final: {df.shape}")


---> Iniciando tratamento da aba: 'Pacientes'
   [✓] Aba 'Pacientes' tratada com sucesso! Formato final: (5000, 8)

---> Iniciando tratamento da aba: 'Dentistas'
   [✓] Aba 'Dentistas' tratada com sucesso! Formato final: (30, 4)

---> Iniciando tratamento da aba: 'Atendimentos'
   [✓] Aba 'Atendimentos' tratada com sucesso! Formato final: (23074, 10)


In [21]:
# EXPORTAÇÃO DOS DADOS TRATADOS
# Finalidade: Salvar os dados limpos em um novo arquivo Excel com abas organizadas.
# -----------------------------------------------------------------------------
caminho_saida = 'Base_Clinica_Odontologica_Felipe_Tratada.xlsx'

with pd.ExcelWriter(caminho_saida, engine='openpyxl') as writer:
    for nome_aba, df_limpo in dados_tratados.items():
        # index=False evita criar uma coluna extra de índices numéricos do Pandas
        df_limpo.to_excel(writer, sheet_name=nome_aba, index=False)


print(f"Processo concluído! Arquivo tratado salvo em:\n--> {caminho_saida}")


Processo concluído! Arquivo tratado salvo em:
--> Base_Clinica_Odontologica_Felipe_Tratada.xlsx
